# 🛡️ SentinelAI — Universal Living Neural ASI Training (SA-MoE)

This notebook trains **SentinelAI** as a Universal Artificial Superintelligence (ASI) substrate:
- **Lossless Byte-Pair Encoding (BPE)** trained on Universal Multi-Domain Corpus (Logic, Math, Systems, Science, Philosophy).
- **Sentinel Adaptive MoE (SA-MoE)** with Shared Expert, Learned Neural Router, and Cross-Expert Residuals.
- **Rotary Positional Embeddings (RoPE)** & **Grouped-Query Attention (GQA)**.
- **Multi-GPU Mixed-Precision (FP16)** training with automatic accelerator fallback.
- **Autoregressive Text Generation** across general reasoning domains.

In [ ]:
# Cell 1: Environment & Accelerator Verification
import os
import sys
import math
import time
import json
from collections import Counter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Detected GPUs: {n_gpus}")
for i in range(n_gpus):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")

In [ ]:
# Cell 2: Flawless Lossless Byte-Pair Encoding (BPE) From Scratch
class BytePairTokenizer:
    """Pure Python Byte-Pair Encoding tokenizer with exact byte-level fidelity."""
    def __init__(self):
        self.special_tokens = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2, "<UNK>": 3, "<MASK>": 4}
        self.inv_special_tokens = {v: k for k, v in self.special_tokens.items()}
        # Base vocab: IDs 5 to 260 map to bytes 0 to 255
        self.vocab = {i + 5: bytes([i]) for i in range(256)}
        self.merges = {}
        self._next_id = 261

    @property
    def vocab_size(self) -> int:
        return self._next_id

    def __len__(self):
        return self._next_id

    def train(self, corpus: str, vocab_size: int = 1024):
        print(f"[BPE] Training tokenizer up to {vocab_size} vocabulary size...")
        num_merges = vocab_size - self.vocab_size
        if num_merges <= 0:
            return
        byte_stream = corpus.encode('utf-8')
        ids = [b + 5 for b in byte_stream]
        for _ in range(num_merges):
            if len(ids) < 2:
                break
            pairs = zip(ids, ids[1:])
            counts = Counter(pairs)
            if not counts:
                break
            best_pair = counts.most_common(1)[0][0]
            new_id = self._next_id
            self._next_id += 1
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            new_ids = []
            i = 0
            while i < len(ids):
                if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                    new_ids.append(new_id)
                    i += 2
                else:
                    new_ids.append(ids[i])
                    i += 1
            ids = new_ids
        print(f"[BPE] Vocab built: {self.vocab_size} total tokens.")

    def encode(self, text: str) -> list[int]:
        byte_stream = text.encode('utf-8')
        ids = [b + 5 for b in byte_stream]
        while len(ids) >= 2:
            pairs = list(zip(ids, ids[1:]))
            pair_to_merge = None
            lowest_id = float('inf')
            for p in pairs:
                if p in self.merges and self.merges[p] < lowest_id:
                    lowest_id = self.merges[p]
                    pair_to_merge = p
            if pair_to_merge is None:
                break
            new_ids = []
            i = 0
            while i < len(ids):
                if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair_to_merge:
                    new_ids.append(self.merges[pair_to_merge])
                    i += 2
                else:
                    new_ids.append(ids[i])
                    i += 1
            ids = new_ids
        return ids

    def decode(self, ids: list[int]) -> str:
        b = bytearray()
        for i in ids:
            if i in self.inv_special_tokens:
                pass
            elif i in self.vocab:
                b.extend(self.vocab[i])
        return b.decode('utf-8', errors='replace')

# Verify tokenizer lossless roundtrip
test_tok = BytePairTokenizer()
test_str = "def solve(x, y): return x + y"
test_tok.train(test_str * 5, vocab_size=280)
assert test_tok.decode(test_tok.encode(test_str)) == test_str, 'Tokenizer verification error!'
print("[BPE] Tokenizer verification: 100% Lossless Roundtrip Confirmed.")

In [ ]:
# Cell 3: Universal Multi-Domain Generalist Corpus
universal_corpus = (
    '# Universal Knowledge & Cognitive Substrate for SentinelAI\n\n'
    '# Section 1: Mathematics, Logic, and Proofs\n'
    'def solve_quadratic(a: float, b: float, c: float):\n'
    '    discriminant = b**2 - 4*a*c\n'
    '    if discriminant < 0:\n'
    '        return None\n'
    '    sqrt_d = discriminant ** 0.5\n'
    '    return ((-b + sqrt_d) / (2*a), (-b - sqrt_d) / (2*a))\n\n'
    'def gcd(a: int, b: int) -> int:\n'
    '    while b != 0:\n'
    '        a, b = b, a % b\n'
    '    return a\n\n'
    '# Section 2: Universal Algorithms & Data Structures\n'
    'class BinarySearchTree:\n'
    '    def __init__(self, val=0):\n'
    '        self.val = val\n'
    '        self.left = None\n'
    '        self.right = None\n\n'
    '    def insert(self, val: int):\n'
    '        if val < self.val:\n'
    '            self.left = self.left.insert(val) if self.left else BinarySearchTree(val)\n'
    '        elif val > self.val:\n'
    '            self.right = self.right.insert(val) if self.right else BinarySearchTree(val)\n'
    '        return self\n\n'
    'def quicksort(arr: list) -> list:\n'
    '    if len(arr) <= 1:\n'
    '        return arr\n'
    '    pivot = arr[len(arr) // 2]\n'
    '    left = [x for x in arr if x < pivot]\n'
    '    middle = [x for x in arr if x == pivot]\n'
    '    right = [x for x in arr if x > pivot]\n'
    '    return quicksort(left) + middle + quicksort(right)\n\n'
    '# Section 3: Epistemology, Scientific Method, and Intelligence\n'
    'The scientific method advances through rigorous empirical falsification.\n'
    'A theory is validated not by decree, but by reproducible experimental proof.\n'
    'Artificial Superintelligence is characterized by recursive self-improvement,\n'
    'wherein an agent systematically analyzes, optimizes, and evolves its own substrate.\n\n'
    '# Section 4: General Systems & Problem Solving\n'
    'def dijkstra(graph: dict, start_node: str) -> dict:\n'
    '    distances = {node: float("inf") for node in graph}\n'
    '    distances[start_node] = 0\n'
    '    unvisited = set(graph.keys())\n'
    '    while unvisited:\n'
    '        curr = min(unvisited, key=lambda n: distances[n])\n'
    '        unvisited.remove(curr)\n'
    '        for neighbor, weight in graph[curr].items():\n'
    '            new_dist = distances[curr] + weight\n'
    '            if new_dist < distances[neighbor]:\n'
    '                distances[neighbor] = new_dist\n'
    '    return distances\n'
) * 20  # Replicate across multi-domain tokens

tokenizer = BytePairTokenizer()
tokenizer.train(universal_corpus, vocab_size=1024)
encoded_data = tokenizer.encode(universal_corpus)
print(f"Universal Corpus encoded: {len(encoded_data):,} tokens | Vocab: {len(tokenizer)}")

In [ ]:
# Cell 4: Universal Sentinel Adaptive MoE (SA-MoE) Transformer Backbone
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, theta: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len: int, device):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        cos = freqs.cos().repeat_interleave(2, dim=-1)
        sin = freqs.sin().repeat_interleave(2, dim=-1)
        return cos, sin

def apply_rotary(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    rotated = torch.stack((-x2, x1), dim=-1).flatten(-2)
    return (x * cos) + (rotated * sin)

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

class CausalGQAAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        cos = cos[:T, :].unsqueeze(0).unsqueeze(0)
        sin = sin[:T, :].unsqueeze(0).unsqueeze(0)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)

        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn = attn.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(attn)

class SwiGLUExpert(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class SentinelAdaptiveMoE(nn.Module):
    def __init__(self, d_model: int, d_ff: int, n_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.shared_expert = SwiGLUExpert(d_model, d_ff)
        self.experts = nn.ModuleList([SwiGLUExpert(d_model, d_ff) for _ in range(n_experts)])
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.cross_residual = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape
        flat_x = x.view(-1, C)
        router_logits = self.router(flat_x)
        router_probs = F.softmax(router_logits, dim=-1)

        topk_probs, topk_indices = torch.topk(router_probs, self.top_k, dim=-1)
        topk_weights = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

        out = self.shared_expert(flat_x)
        for i in range(self.top_k):
            idx = topk_indices[:, i]
            w = topk_weights[:, i].unsqueeze(-1)
            for e_idx, expert in enumerate(self.experts):
                mask = (idx == e_idx)
                if mask.any():
                    out[mask] += w[mask] * expert(flat_x[mask])

        # Load balancing auxiliary loss
        density = router_probs.mean(dim=0)
        aux_loss = self.n_experts * torch.sum(density * router_probs.mean(dim=0))
        final_out = out.view(B, T, C) + self.cross_residual(x)
        return final_out, aux_loss

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, d_ff: int, n_experts: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalGQAAttention(d_model, n_heads, n_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.moe = SentinelAdaptiveMoE(d_model, d_ff, n_experts=n_experts, top_k=2)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
        x = x + self.attn(self.norm1(x), cos, sin)
        moe_out, aux_loss = self.moe(self.norm2(x))
        x = x + moe_out
        return x, aux_loss

class SentinelTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, n_layers: int = 6, n_heads: int = 8, n_kv_heads: int = 4, d_ff: int = 1024, n_experts: int = 4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.rope = RotaryEmbedding(d_model // n_heads)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, d_ff, n_experts) for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor, targets: torch.Tensor = None):
        B, T = x.shape
        h = self.tok_embed(x)
        cos, sin = self.rope(T, x.device)
        total_aux = 0.0
        for layer in self.layers:
            h, aux = layer(h, cos, sin)
            total_aux = total_aux + aux
        h = self.final_norm(h)
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))
        return logits, loss, total_aux

    @torch.no_grad()
    def generate(self, prompt_ids: torch.Tensor, max_new_tokens: int = 50, temperature: float = 0.8, top_k: int = 40):
        self.eval()
        for _ in range(max_new_tokens):
            x_cond = prompt_ids[:, -512:]
            logits, _, _ = self(x_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            prompt_ids = torch.cat((prompt_ids, next_tok), dim=1)
        return prompt_ids

In [ ]:
# Cell 5: Universal Chunk Dataset & DataLoader
class UniversalChunkDataset(Dataset):
    def __init__(self, token_ids: list, seq_len: int = 128):
        self.tokens = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        self.n_chunks = max(0, (len(self.tokens) - seq_len - 1) // seq_len + 1)

    def __len__(self):
        return self.n_chunks

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.tokens[start : start + self.seq_len]
        y = self.tokens[start + 1 : start + self.seq_len + 1]
        return x, y

dataset = UniversalChunkDataset(encoded_data, seq_len=128)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, drop_last=True)
print(f"Created Universal DataLoader with {len(dataset)} sequence chunks.")

In [ ]:
# Cell 6: Mixed-Precision Training Loop
model = SentinelTransformer(
    vocab_size=len(tokenizer),
    d_model=256,
    n_layers=6,
    n_heads=8,
    n_kv_heads=4,
    d_ff=1024,
    n_experts=4
)
param_count = sum(p.numel() for p in model.parameters())
print(f"SentinelAI Model Parameters: {param_count:,} ({param_count/1e6:.2f}M)")

if torch.cuda.device_count() > 1:
    print(f"DataParallel initialized across {torch.cuda.device_count()} GPUs!")
    parallel_model = nn.DataParallel(model).to(device)
else:
    parallel_model = model.to(device)

optimizer = torch.optim.AdamW(parallel_model.parameters(), lr=4e-4, betas=(0.9, 0.95), weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

epochs = 12
loss_history = []
print(f"Starting Universal training for {epochs} epochs on {device}...")
t0 = time.time()

parallel_model.train()
step = 0
for epoch in range(epochs):
    epoch_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits, loss, aux = parallel_model(bx, targets=by)
            if loss.ndim > 0:
                loss = loss.mean()
            if isinstance(aux, torch.Tensor) and aux.ndim > 0:
                aux = aux.mean()
            total_loss = loss + 0.01 * aux

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(parallel_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        step += 1
        loss_val = loss.item()
        loss_history.append(loss_val)
        epoch_loss += loss_val

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Avg Loss: {avg_loss:.4f} | Time: {time.time()-t0:.1f}s")

print("\n[DONE] Universal Training finished successfully!")

In [ ]:
# Cell 7: Plot Training Loss Convergence
plt.figure(figsize=(10, 4))
plt.plot(loss_history, label="Cross-Entropy Loss", color="#00ff88")
plt.title("SentinelAI Universal Training Loss Convergence (SA-MoE)")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Cell 8: Universal Inference & General Problem Solving
universal_prompts = [
    "def solve_quadratic(",
    "class BinarySearchTree:",
    "The scientific method advances through",
    "def dijkstra("
]

model.eval()
print("=== SentinelAI General Problem Solving Evaluation ===\n")
for p in universal_prompts:
    input_ids = torch.tensor([tokenizer.encode(p)], device=device)
    gen_ids = model.generate(input_ids, max_new_tokens=45, temperature=0.7, top_k=20)
    result_text = tokenizer.decode(gen_ids[0].tolist())
    print(f"> Input Prompt: {p}")
    print(f"> Generated Output:\n{result_text}\n{"-*-"*20}")

In [ ]:
# Cell 9: Save Checkpoint for Universal ASI Deployment
output_path = "/kaggle/working/sentinel_final.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': tokenizer.vocab,
    'merges': tokenizer.merges,
    'd_model': 256,
    'n_layers': 6,
    'n_heads': 8,
    'n_experts': 4
}, output_path)
print(f"[SAVED] Universal Checkpoint exported to {output_path}")